# Complete Fake News Dataset Crawler
Upload `url_fake_1.xlsx`, then run all cells. Output will include CSV and XLSX with full columns.

In [ ]:
!pip -q install beautifulsoup4 requests pandas openpyxl


In [ ]:

# ============================================================
# COMPLETE FAKE NEWS DATASET FROM URL LIST
# Input : url_fake_1.xlsx, must contain column "url"
# Output: url_fake_1_completed_full.csv and .xlsx
# Columns:
# id, title, content, source, url, publish_date, topic, label, explanation
# ============================================================

import re
import time
import uuid
import random
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup


INPUT_FILE = "url_fake_1.xlsx"
OUTPUT_CSV = "url_fake_1_completed_full.csv"
OUTPUT_XLSX = "url_fake_1_completed_full.xlsx"


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

SOURCE_MAP = {
    "vnexpress.net": "vnexpress",
    "vietnamnet.vn": "vietnamnet",
    "baotintuc.vn": "baotintuc",
    "baoquocte.vn": "baoquocte",
    "baomoi.com": "baomoi",
    "giadinh.suckhoedoisong.vn": "giadinh_suckhoedoisong",
    "suckhoedoisong.vn": "suckhoedoisong",
    "doanhnghiepvn.vn": "doanhnghiepvn",
    "kienthuc.net.vn": "kienthuc",
    "dantri.com.vn": "dantri",
    "tuoitre.vn": "tuoitre",
    "thanhnien.vn": "thanhnien",
    "laodong.vn": "laodong",
    "cafef.vn": "cafef",
    "kenh14.vn": "kenh14",
    "zingnews.vn": "zingnews",
    "vtcnews.vn": "vtcnews",
}


def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_domain(url):
    domain = urlparse(str(url)).netloc.lower()
    domain = domain.replace("www.", "")
    return domain


def get_source(url):
    domain = normalize_domain(url)
    if domain in SOURCE_MAP:
        return SOURCE_MAP[domain]
    parts = domain.split(".")
    return parts[0] if parts else ""


def fetch_html(url, retries=3, timeout=20):
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=timeout)
            if response.status_code == 200 and len(response.text) > 500:
                response.encoding = response.apparent_encoding
                return response.text
            print(f"[WARN] {response.status_code}: {url}")
        except Exception as e:
            print(f"[WARN] Fetch failed attempt {attempt + 1}/{retries}: {url} -> {e}")
        time.sleep(1.5 + attempt + random.random())
    return ""


def get_meta(soup, names):
    for name in names:
        tag = soup.find("meta", attrs={"property": name}) or soup.find("meta", attrs={"name": name})
        if tag and tag.get("content"):
            return clean_text(tag.get("content"))
    return ""


def extract_title(soup):
    title = get_meta(soup, ["og:title", "twitter:title"])
    if title:
        return title

    h1 = soup.find("h1")
    if h1:
        return clean_text(h1.get_text(" "))

    if soup.title:
        return clean_text(soup.title.get_text(" "))

    return ""


def extract_publish_date(soup, html):
    date = get_meta(
        soup,
        [
            "article:published_time",
            "pubdate",
            "publishdate",
            "date",
            "dc.date",
            "DC.date.issued",
            "dcterms.created",
            "sailthru.date",
        ],
    )
    if date:
        return date

    time_tag = soup.find("time")
    if time_tag:
        if time_tag.get("datetime"):
            return clean_text(time_tag.get("datetime"))
        return clean_text(time_tag.get_text(" "))

    text = clean_text(soup.get_text(" "))
    patterns = [
        r"\b\d{1,2}:\d{2}\s+\d{1,2}/\d{1,2}/\d{4}\b",
        r"\b\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}\b",
        r"\b\d{1,2}/\d{1,2}/\d{4}\b",
        r"\b\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}",
        r"(Thứ|Chủ nhật)[^,]{0,30},?\s*\d{1,2}[:h]\d{2}[^0-9]{0,20}\d{1,2}/\d{1,2}/\d{4}",
    ]
    for p in patterns:
        m = re.search(p, text, flags=re.IGNORECASE)
        if m:
            return clean_text(m.group(0))
    return ""


def extract_topic(soup, url):
    topic = get_meta(soup, ["article:section", "section", "category", "news_keywords"])
    if topic:
        return clean_text(topic.split(",")[0])

    # Try breadcrumb/category links
    candidates = []
    for selector in [
        ".breadcrumb a",
        ".breadcrumbs a",
        ".cate a",
        ".category a",
        ".menu a",
        "nav a",
    ]:
        for a in soup.select(selector):
            txt = clean_text(a.get_text(" "))
            if 2 <= len(txt) <= 40 and txt.lower() not in ["trang chủ", "home", "mới nhất"]:
                candidates.append(txt)

    for c in candidates:
        if c.lower() not in ["facebook", "youtube", "tiktok", "twitter"]:
            return c

    # Fallback by URL path
    u = url.lower()
    rules = [
        ("Giáo dục", ["giao-duc", "hoc-duong", "du-hoc", "tuyen-sinh"]),
        ("Pháp luật", ["phap-luat", "an-ninh", "toa-an", "toi-pham", "cong-an"]),
        ("Chính trị", ["chinh-tri", "quoc-hoi", "chinh-phu"]),
        ("Sức khỏe", ["suc-khoe", "y-te", "benh", "song-khoe"]),
        ("Kinh tế", ["kinh-te", "thi-truong", "tai-chinh", "doanh-nghiep", "bat-dong-san"]),
        ("Thế giới", ["the-gioi", "quoc-te", "phan-tichnhan-dinh", "nga", "trung-quoc", "my-"]),
        ("Giải trí", ["giai-tri", "showbiz", "phim", "sao"]),
        ("Đời sống", ["doi-song", "tam-su", "gia-dinh", "chuyen-vo-chong"]),
        ("Tử vi - phong thủy", ["tu-vi", "con-giap", "phong-thuy", "ngay-sinh-am-lich"]),
        ("Thể thao", ["the-thao", "bong-da"]),
    ]
    for label, keys in rules:
        if any(k in u for k in keys):
            return label
    return "Tin tức"


def extract_content(soup):
    for tag in soup(["script", "style", "noscript", "iframe", "svg"]):
        tag.decompose()

    # Remove obvious unrelated areas
    for selector in [
        "header", "footer", "nav",
        ".related", ".tin-lien-quan", ".comment", ".comments",
        ".social", ".share", ".ads", ".advertisement",
        ".box", ".sidebar", ".menu", ".breadcrumb"
    ]:
        for tag in soup.select(selector):
            tag.decompose()

    article_selectors = [
        "article",
        ".article-content",
        ".detail-content",
        ".content-detail",
        ".news-content",
        ".maincontent",
        ".main-content",
        ".detail",
        ".body-content",
        ".entry-content",
        ".post-content",
        ".dt-news__content",
        "#main-detail",
        "#content_detail",
        "#article_body",
    ]

    best_text = ""
    for selector in article_selectors:
        blocks = soup.select(selector)
        for block in blocks:
            paragraphs = []
            for p in block.find_all(["p", "h2"], recursive=True):
                txt = clean_text(p.get_text(" "))
                if len(txt) >= 30:
                    paragraphs.append(txt)
            text = " ".join(paragraphs)
            if len(text) > len(best_text):
                best_text = text

    if len(best_text) >= 200:
        return best_text

    # Generic fallback: collect all meaningful paragraphs
    paragraphs = []
    for p in soup.find_all("p"):
        txt = clean_text(p.get_text(" "))
        if len(txt) >= 40:
            paragraphs.append(txt)

    # Deduplicate while preserving order
    seen = set()
    unique = []
    for p in paragraphs:
        key = p[:120].lower()
        if key not in seen:
            seen.add(key)
            unique.append(p)

    return " ".join(unique)


def summarize_for_explanation(title, content, max_len=220):
    base = content if content else title
    base = clean_text(base)
    if len(base) <= max_len:
        return base
    return base[:max_len].rsplit(" ", 1)[0] + "..."


def make_explanation(title, content, topic, url):
    text = f"{title} {content}".lower()
    summary = summarize_for_explanation(title, content)

    reasons = []

    # Important: this dataset is provided as a fake URL list, so explanation uses that label source.
    reasons.append("URL này thuộc danh sách dữ liệu FAKE đầu vào nên được giữ nhãn FAKE trong bộ dữ liệu.")

    if any(k in text for k in ["con giáp", "tử vi", "phong thủy", "ngày sinh âm lịch", "tài lộc", "vận may"]):
        reasons.append(
            "Nội dung mang tính dự đoán/tâm linh như tử vi, con giáp, vận may hoặc tài lộc; "
            "các khẳng định kiểu này thường không có cơ sở kiểm chứng khoa học hoặc bằng chứng dữ liệu rõ ràng."
        )

    if any(k in text for k in ["tâm sự", "chồng", "vợ", "ly hôn", "người yêu", "mẹ chồng", "ngoại tình"]):
        reasons.append(
            "Bài viết có dạng câu chuyện cá nhân/tâm sự, nhân vật và tình tiết khó xác minh độc lập; "
            "nội dung dễ tạo cảm xúc và giật tít hơn là cung cấp bằng chứng kiểm chứng được."
        )

    if any(k in text for k in ["sức khỏe", "bệnh", "thuốc", "chữa", "ung thư", "vaccine", "giảm cân"]):
        reasons.append(
            "Bài có yếu tố sức khỏe/y tế nên cần dẫn chứng từ cơ quan chuyên môn hoặc nghiên cứu đáng tin cậy; "
            "nếu thiếu nguồn kiểm chứng rõ ràng thì không nên xem là thông tin xác thực."
        )

    if any(k in text for k in ["trump", "mỹ", "nga", "trung quốc", "iran", "nato", "ukraine", "chiến tranh", "tổng thống"]):
        reasons.append(
            "Bài liên quan chính trị/quốc tế, nhóm thông tin dễ bị diễn giải lệch hoặc giật tít; "
            "cần đối chiếu với nguồn chính thống/nguồn gốc phát ngôn trước khi dùng làm tin thật."
        )

    if any(k in text for k in ["sốc", "choáng", "bất ngờ", "không ngờ", "tiền tỷ", "sững sờ", "lạ lùng"]):
        reasons.append(
            "Tiêu đề chứa yếu tố câu kéo cảm xúc như sốc/choáng/bất ngờ/tiền tỷ, làm tăng rủi ro clickbait "
            "và không đủ để kết luận độ xác thực của thông tin."
        )

    if len(content) < 250:
        reasons.append(
            "Nội dung crawl được khá ngắn hoặc thiếu phần thân bài, vì vậy mẫu cần được kiểm chứng thêm trước khi dùng làm dữ liệu tin thật."
        )

    # Avoid only generic explanation
    if len(reasons) == 1:
        reasons.append(
            "Phần nội dung cần được đối chiếu thêm với bằng chứng độc lập; trong bài chưa thể hiện rõ cơ sở kiểm chứng đủ mạnh để đổi sang REAL."
        )

    return (
        f"Bài viết được gán nhãn FAKE. Chủ đề chính: {topic}. "
        f"Tóm tắt nội dung: {summary} "
        + " ".join(reasons)
    )


def parse_article(url):
    html = fetch_html(url)
    if not html:
        title = ""
        content = ""
        publish_date = ""
        topic = ""
    else:
        soup = BeautifulSoup(html, "html.parser")
        title = extract_title(soup)
        content = extract_content(soup)
        publish_date = extract_publish_date(soup, html)
        topic = extract_topic(soup, url)

    source = get_source(url)

    # In case title/content is still missing
    if not title:
        slug = urlparse(url).path.split("/")[-1]
        slug = re.sub(r"\.(html|htm|epi)$", "", slug)
        slug = re.sub(r"-\d+$", "", slug)
        title = clean_text(slug.replace("-", " "))

    if not topic:
        topic = "Tin tức"

    explanation = make_explanation(title, content, topic, url)

    return {
        "id": str(uuid.uuid4()),
        "title": title,
        "content": content,
        "source": source,
        "url": url,
        "publish_date": publish_date,
        "topic": topic,
        "label": "FAKE",
        "explanation": explanation,
    }


def main():
    df_url = pd.read_excel(INPUT_FILE)
    df_url.columns = [str(c).strip().lower() for c in df_url.columns]

    if "url" not in df_url.columns:
        raise ValueError("File input phải có cột tên là 'url'.")

    urls = (
        df_url["url"]
        .dropna()
        .astype(str)
        .map(str.strip)
        .loc[lambda s: s.str.startswith("http")]
        .drop_duplicates()
        .tolist()
    )

    rows = []
    for i, url in enumerate(urls, start=1):
        print(f"[{i}/{len(urls)}] Crawling: {url}")
        row = parse_article(url)
        rows.append(row)

        # Save checkpoint every 25 rows
        if i % 25 == 0:
            temp_df = pd.DataFrame(rows)
            temp_df.to_csv("checkpoint_url_fake_completed.csv", index=False, encoding="utf-8-sig")

        time.sleep(random.uniform(0.5, 1.5))

    final_df = pd.DataFrame(rows, columns=[
        "id", "title", "content", "source", "url",
        "publish_date", "topic", "label", "explanation"
    ])

    final_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    final_df.to_excel(OUTPUT_XLSX, index=False)

    print("DONE")
    print("Rows:", len(final_df))
    print("CSV:", OUTPUT_CSV)
    print("XLSX:", OUTPUT_XLSX)


if __name__ == "__main__":
    main()


[1/623] Crawling: https://giadinh.suckhoedoisong.vn/chong-cu-bat-ngo-xuat-hien-sau-5-nam-ly-hon-toi-sung-so-khi-anh-ta-di-xe-o-to-tien-ty-va-dua-ra-loi-de-nghi-172250223162313132.htm
[2/623] Crawling: https://baoquocte.vn/ru-nhau-tap-tran-chung-iran-nga-va-trung-quoc-muon-day-lui-suc-anh-huong-cua-my-306938.html
[3/623] Crawling: https://baotintuc.vn/phan-tichnhan-dinh/trung-quoc-pha-vo-chien-luoc-thay-the-nguon-cung-cap-khi-dot-nga-cua-phuong-tay-20250309111306555.htm
[WARN] Fetch failed attempt 1/3: https://baotintuc.vn/phan-tichnhan-dinh/trung-quoc-pha-vo-chien-luoc-thay-the-nguon-cung-cap-khi-dot-nga-cua-phuong-tay-20250309111306555.htm -> HTTPSConnectionPool(host='baotintuc.vn', port=443): Max retries exceeded with url: /phan-tichnhan-dinh/trung-quoc-pha-vo-chien-luoc-thay-the-nguon-cung-cap-khi-dot-nga-cua-phuong-tay-20250309111306555.htm (Caused by SSLError(SSLError(1, '[SSL: UNSAFE_LEGACY_RENEGOTIATION_DISABLED] unsafe legacy renegotiation disabled (_ssl.c:1010)')))
[WARN] Fetc